# 1. Import data / read headline

In [7]:
# imports (this is all you need for this assignment - please, do not import anything else)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
import os


# if the present working directory is the directory of this file, change the path to the project root
if os.getcwd().endswith("notebooks"):
    os.chdir("..")

In [10]:
## Load the data
DATA_FILE = os.path.join( "data/001MoDe_R1.csv")
MARKER_FILE = os.path.join("data/001MoDe_R1.marker.csv")

In [ ]:
#define a function to read the header of the data file and return the column names
def read_header(filename):
    # Return the task parameters of the first line as a dictionary
    with open(filename) as f:
        first_line = f.readline().strip() # Read the first line and strip whitespace
    header = {}
    for item in first_line.split(";"): # Split the line by semicolon to get key-value combinations
        key, value = item.split(" ", 1)# Split each combination into key and value (only at the first space)
        # Try to convert the value to a float, if it fails, keep it as a string
        try:
            value = float(value)
        except ValueError:
            pass  # keep text values (colors, software name...) as strings
        header[key] = value
    return header



# Read the header of the data file and store the header in a dictionary
header = read_header(DATA_FILE)
print(f"Header line information in :")
#print the task paramaeters in a readable format
for key, value in header.items():
    print(f"  {key}: {value}")

Header line information in 'data/001MoDe_R1.csv':
  screenWidth: 1104.0
  screenHeight: 661.0
  cornerX: 302.0
  cornerY: 80.0
  centerX: 552.0
  centerY: 330.0
  externalRadius: 250.0
  internalRadius: 170.0
  borderRadius: 1.0
  cursorRadius: 16.0
  indexOfDifficulty: 28.00696429476858
  taskRadius: 209.5
  taskTolerance: 47.0
  borderColor: java.awt.Color[r=255,g=255,b=255]
  backgroundColor: java.awt.Color[r=0,g=0,b=0]
  cursorColorRecord: java.awt.Color[r=255,g=0,b=0]
  cursorColorWait: java.awt.Color[r=255,g=255,b=0]
  autoStart: 3600.0
  cycleMaxNumber: 6.0
  cycleDuration: 20.0
  software: LSL-mouse
  version: 1.2.0rc5
  task: CircularTarget
  isWithLSL: false


In [22]:
def read_markers(filename):
    # Return the list of timestamp_ms, message and the list of the summary table lines
    with open(filename) as f:
        lines = f.read().splitlines()
    markers, table_lines = [], []
    for line in lines[3:]:  # skip header, date and empty line
        parts = line.split(",", 2)
        if len(parts) == 3 and parts[1].strip().isdigit():
            markers.append((int(parts[1]), parts[2]))
        elif line.strip() != "":
            table_lines.append(line)
    return markers, table_lines


#call the function and print the markers and the summary table lines
markers, table_lines = read_markers(MARKER_FILE)
print(f"Markers in '{MARKER_FILE}':")
for ts, msg in markers: # for every marker, print the timestamp and the message
    print(f"{ts}: {msg}")
for line in table_lines: #same for the summary table lines
    print(line)




def get_record_times(markers):
    # Return an array (n_records, 2) with [start, end] timestamps (ms) of each record
    times = []
    for i, (ts, msg) in enumerate(markers): # enumerate over the markers to get the index and the (timestamp, message) 
        if "DoRecord" in msg and i + 1 < len(markers): # check if the message contains "DoRecord" and if there is a next marker then stock the start timestamp and the next marker's timestamp as the end timestamp to isolate a record
            times.append([ts, markers[i + 1][0]])
    return np.array(times, dtype=np.int64)

#call the function and print the record times
record_times = get_record_times(markers)
print("record times (ms):")
for start, end in record_times:
    print(f"{start} to {end} ({(end - start) / 1000:.3f} s)")

Markers in 'data/001MoDe_R1.marker.csv':
1616776712325: KeyTyped=32 DoCycleChange
1616776712331: DoCycleChange:DoStartCycleTimedSequence
1616776712333: DoCycleChange:DoRecord RecordDone=0 PauseDone=0 ToDo=6
1616776732336: DoCycleChange:DoPause RecordDone=1 PauseDone=0 ToDo=6
1616776752340: DoCycleChange:DoRecord RecordDone=1 PauseDone=1 ToDo=6
1616776772351: DoCycleChange:DoPause RecordDone=2 PauseDone=1 ToDo=6
1616776792372: DoCycleChange:DoRecord RecordDone=2 PauseDone=2 ToDo=6
1616776812386: DoCycleChange:DoPause RecordDone=3 PauseDone=2 ToDo=6
1616776832390: DoCycleChange:DoRecord RecordDone=3 PauseDone=3 ToDo=6
1616776852404: DoCycleChange:DoPause RecordDone=4 PauseDone=3 ToDo=6
1616776872419: DoCycleChange:DoRecord RecordDone=4 PauseDone=4 ToDo=6
1616776892433: DoCycleChange:DoPause RecordDone=5 PauseDone=4 ToDo=6
1616776898364: KeyTyped=113 WINDOW_CLOSING
1616776898367: 
    Var , nLaps ,      Re ,      Te ,   error ,  MT/lap , IDe/lap ,      Be ,     IPe ,
   unit ,   lap ,   p